# Hybrid 검색 20분 — Dense 가 놓친 것을 글자로 잡는다

4회차 수업 1~2부에서 씁니다. 셀 번호는 코드 셀 첫 줄의 `# [셀 N]` 표시입니다. 메뉴 Kernel → Restart Kernel and Run All Cells 를 하면 왼쪽의 `[N]` 도 같은 번호가 됩니다.

**이 노트북이 끝나면**

- 3회차 baseline 이 어떤 문항을 놓쳤는지, 놓친 문항의 공통점이 무엇인지 말할 수 있습니다.
- 공백 토크나이저와 Kiwi 형태소 토크나이저가 같은 문장을 어떻게 다르게 자르는지 압니다. 이것이 **실습 1** 의 재료입니다.
- BM25 가 조항번호 질문을 어떻게 잡는지, RRF 가 두 순위를 어떻게 합치는지(**실습 2**) 압니다.

**필요한 것**: 3회차에 만든 `data/chunks/chunks.jsonl` 과 `results/baseline.json`. **필요 없는 것**: API 키, 임베딩 모델, 리랭커. 이 노트북은 벡터 DB(Qdrant)를 열지 않으므로, 터미널에서 `make hybrid` 를 돌리는 동안 열어 두어도 충돌하지 않습니다.

> **왜 검색을 두 가지로 하나.** 임베딩은 뜻이 비슷한 것을 찾습니다. "제31조"와 "제32조"는 뜻이 거의 같아서 임베딩으로는 구별이 안 됩니다. 글자가 같은 것을 찾는 검색(BM25)을 옆에 두고, 두 검색의 순위를 합치는 것이 오늘 하는 일입니다.

In [5]:
# [셀 1] 준비
import json
import sys
from importlib.metadata import version
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
for pkg in ("kiwipiepy", "rank-bm25"):
    print(f"{pkg:12}", version(pkg))

chunks = [json.loads(l) for l in (ROOT / "data/chunks/chunks.jsonl").open(encoding="utf-8") if l.strip()]
by_id = {c["chunk_id"]: c for c in chunks}
baseline = json.loads((ROOT / "results/baseline.json").read_text(encoding="utf-8"))
seed = {q["id"]: q for q in (json.loads(l) for l in (ROOT / "data/golden/seed_golden.jsonl").open(encoding="utf-8") if l.strip())}
gold = json.loads((ROOT / "data/golden/gold_chunk_ids.json").read_text(encoding="utf-8"))
o = baseline["overall"]
print(f"청크 {len(chunks):,}개 · baseline Recall@5 {o['recall@5']:.1%} · MRR {o['mrr']:.3f}")

kiwipiepy    0.24.0
rank-bm25    0.2.2
청크 3,689개 · baseline Recall@5 77.5% · MRR 0.553


## 1. baseline 이 놓친 문항

3회차 `make baseline` 의 문항별 결과입니다. 상위 5개 안에 정답 청크가 없었던 문항만 뽑아, 정답 청크와 baseline 이 1등으로 고른 청크를 나란히 놓습니다. 질문에 무엇이 들어 있는지 보세요. 조항번호, 상품명, 회사 이름처럼 **글자가 정확히 같아야 하는 것**이 많습니다. 임베딩은 이런 것을 잘 구별하지 못합니다.

In [6]:
# [셀 2] baseline 이 상위 5개 안에 정답을 못 넣은 문항
def label(cid):
    c = by_id.get(cid)
    return f"{c['doc_id']} {c.get('article') or '-'}" if c else (cid or "-")

missed = [r for r in baseline["rows"] if r["recall@5"] == 0]
print(f"놓친 문항 {len(missed)}/{baseline['n']}\n")
for r in missed:
    q = seed[r["id"]]
    print(f"[{r['id']} {r['type']}] {q['question']}")
    print(f"   정답: {label(gold.get(r['id'], ['-'])[0])}")
    print(f"   baseline 1등: {label(r['top_chunk'])}\n")

놓친 문항 9/40

[S13 상품명조건형] 카카오뱅크 정기예금 특약상 가입대상은 누구인가요?
   정답: kakao_deposit_terms_2024 제3조
   baseline 1등: kakao_deposit_2026_txt -

[S14 상품명조건형] iM뱅크 가계대출은 만기까지 얼마 남지 않으면 중도상환수수료가 면제되나요?
   정답: im_household_2025 -
   baseline 1등: sf_realestate_loan_2026 -

[S15 메타데이터형] 2013년 7월 1일 시행 하나은행 여신거래기본약관(가계용) 기준으로, 기한의 이익 상실 사전 통지는 며칠 전까지였나요?
   정답: hana_credit_terms_2013 제7조
   baseline 1등: fss_credit_terms_2024_pdf 제7조

[S17 메타데이터형] 카카오뱅크 정기예금의 '특약'(약관)에서 정한 계약기간 범위는?
   정답: kakao_deposit_terms_2024 제5조
   baseline 1등: kakao_deposit_2026_txt -

[S33 계산형] iM뱅크 가계대출 1억 2천만원(약정 5%, 가산 3%)의 월이자 50만원을 2개월 연체하면 연체이자는?
   정답: im_household_2025 -
   baseline 1등: im_mortgage_2024 -

[S35 계산형] 현행 실손 표준약관 기본형 급여 통원에서 공제금액은 어떻게 정해지나요?(의원 기준)
   정답: law_silson_std_pdf 제42조
   baseline 1등: law_silson_std_pdf 제5조

[S38 유사상품비교형] 하나은행 여신거래기본약관 제7조 제2항의 기한이익 상실 사전 통지 기한은 2013년 판과 2024년 판에서 어떻게 달라졌나요?
   정답: hana_credit_terms_2013 제7조
   baseline 1등: fss_credit_terms_2024_pdf 제8조

[S39 미답변형] ABL 인터넷암보험Ⅲ를 5년 뒤 

## 2. 토크나이저 — 같은 문장을 다르게 자른다 (실습 1)

BM25 는 토큰이 정확히 같아야 점수를 줍니다. 공백으로 자르면 "제 15 조의 2"가 네 토막이 되고, "중도해지이율은"에 조사 "은"이 붙은 채 남습니다. Kiwi 는 조항을 하나로 모으고 조사·어미를 버립니다.

아래 셀은 실습 1(`src/finrag/index/bm25_ko.py` 의 `tokenize`)을 채우기 전에는 "아직 채우지 않았습니다"가 나옵니다. 채운 뒤 다시 실행합니다.

In [7]:
# [셀 3] 공백 vs Kiwi (실습 1 뒤에 다시 실행)
from finrag.index.bm25_ko import tokenize, whitespace_tokenize

for s in ["제 15 조의 2에 따른 중도해지이율은 연 1.4%입니다",
          "카카오뱅크의 정기예금은 만기 후 이율이 어떻게 되나요",
          "일부결제금액이월약정(리볼빙) 수수료율은 어떻게 되나요"]:
    print(s)
    print("   공백:", whitespace_tokenize(s))
    try:
        print("   Kiwi:", tokenize(s))
        print("   질의 확장:", tokenize(s, expand=True))
    except NotImplementedError as e:
        print("   Kiwi: 아직 채우지 않았습니다 —", e)
    print()

제 15 조의 2에 따른 중도해지이율은 연 1.4%입니다
   공백: ['제', '15', '조의', '2에', '따른', '중도해지이율은', '연', '1.4%입니다']
   Kiwi: 아직 채우지 않았습니다 — TODO: tokenize 를 구현하세요

카카오뱅크의 정기예금은 만기 후 이율이 어떻게 되나요
   공백: ['카카오뱅크의', '정기예금은', '만기', '후', '이율이', '어떻게', '되나요']
   Kiwi: 아직 채우지 않았습니다 — TODO: tokenize 를 구현하세요

일부결제금액이월약정(리볼빙) 수수료율은 어떻게 되나요
   공백: ['일부결제금액이월약정(리볼빙)', '수수료율은', '어떻게', '되나요']
   Kiwi: 아직 채우지 않았습니다 — TODO: tokenize 를 구현하세요



## 3. BM25 로 조항번호 질문을 검색한다

하나카드 표준약관 청크만으로 작은 인덱스를 만들고 "제31조 리볼빙"을 찾습니다. 같은 질문, 같은 청크인데 토크나이저에 따라 1등이 달라지는지 봅니다. 점수는 BM25 점수라 상한이 없습니다(코사인처럼 0~1 이 아닙니다). 이것이 뒤에서 점수를 더하지 않고 순위를 합치는 이유입니다.

In [8]:
# [셀 4] 작은 BM25 인덱스 — 공백 vs Kiwi (실습 1 뒤에 다시 실행)
from finrag.index.bm25_ko import BM25Index

small = [c for c in chunks if c["doc_id"] == "hanacard_std_2017"]
query = "제31조 리볼빙"

def show(name, idx, tok):
    print(f"[{name}] {query}")
    for cid, score in idx.search(query, k=3, tokenizer=tok):
        c = by_id[cid]
        print(f"   {score:6.2f}  {c.get('article') or '-':8} {' '.join(c['text'].split())[:40]}")

show("공백", BM25Index.build(small, tokenizer=whitespace_tokenize), whitespace_tokenize)
try:
    show("Kiwi", BM25Index.build(small, tokenizer=tokenize), tokenize)
except NotImplementedError as e:
    print("[Kiwi] 아직 채우지 않았습니다 —", e)

[공백] 제31조 리볼빙
[Kiwi] 아직 채우지 않았습니다 — TODO: tokenize 를 구현하세요


## 4. RRF — 두 순위를 합친다 (실습 2)

Dense 가 준 순위와 BM25 가 준 순위는 점수 단위가 다릅니다(코사인은 0~1, BM25 는 상한 없음). 그래서 점수를 더하지 않고 **순위**만 씁니다. 순위 r 인 문서에 1/(60+r) 을 주고, 두 목록에 다 나온 문서는 두 몫을 더합니다.

실습 2(`src/finrag/retrieval/hybrid.py` 의 `rrf`)를 채운 뒤 실행합니다.

In [ ]:
# [셀 5] RRF 장난감 (실습 2 뒤에 다시 실행)
from finrag.retrieval.hybrid import rrf

dense_rank = ["A", "B", "C", "D"]     # 임베딩이 고른 순서
bm25_rank = ["C", "A", "E"]           # 키워드가 고른 순서
try:
    fused = rrf([dense_rank, bm25_rank])
    for cid, score in sorted(fused.items(), key=lambda x: -x[1]):
        parts = [f"1/(60+{r.index(cid) + 1})" for r in (dense_rank, bm25_rank) if cid in r]
        print(f"   {cid}  {score:.4f}  = {' + '.join(parts)}")
    print("\nA 는 두 목록에 다 있어서 1등, C 도 둘 다 있어서 2등입니다. 한 목록에만 있는 B·E·D 는 뒤로 갑니다.")
except NotImplementedError as e:
    print("아직 채우지 않았습니다 —", e)

## 5. 오늘 코드에서는 어디에 있나

| 이 노트북 | 오늘 코드 |
|---|---|
| 셀 3 의 `tokenize` | `src/finrag/index/bm25_ko.py` — **실습 1.** 인덱싱할 때와 검색할 때 양쪽에서 쓴다 |
| 셀 4 의 `BM25Index` | 같은 파일. `hybrid._bm25()` 가 청크 3,700개 전체로 만든다 |
| 셀 5 의 `rrf` | `src/finrag/retrieval/hybrid.py` — **실습 2.** `search()` 가 Dense 와 BM25 순위를 여기에 넣는다 |
| (없음) | `src/finrag/retrieval/rerank.py` — Cross-Encoder 리랭커. 완성본. 터미널의 `make hybrid` 에서 돈다 |
| (없음) | `src/finrag/retrieval/filters.py` — 필터와 최신 판본 고르기. `scripts/search_demo.py --latest` |

다음은 터미널입니다. `make bm25-compare`(토크나이저 비교, 40문항), `make hybrid-norerank`, `make hybrid`, `make rerank-sweep`(실습 3).